# Lab 5: Counting Questions

**Workshop 2 · Block 4 · 20 minutes**

Most Level 4 questions are counts. *"How many workshop posters are on the Innovation Academy website for 2025?"* The answer is 31, and no page says 31. Your bot has to find every matching item and count them itself.

Normal retrieval cannot do that. It returns the **k nearest** chunks, so with k = 5 your bot has seen at most five posters, whatever the embeddings do. The fix is to **filter by metadata** (every match, not the nearest few), **list** what was found, then **count**.

The sample corpus has no posters, so in this lab you count something you do have: the **five photos you indexed in lab 4**. The method is exactly the same.

**What you will do**

1. Baseline: ask a counting question the normal way
2. Count with a filter
3. Break it on purpose, to see the k ceiling
4. Design the filter your own bot needs for posters
5. Route, then time it

Then two extensions: **A** retrieve wide and judge (when metadata is missing), **B** a count with two parts.

> **You need:** the five image chunks you indexed in **lab 4, Step 4**. Step 0 checks for them and, if lab 4 saved `data/descriptions.json`, re-indexes them for you.
>
> **You leave with:** a working `answer_by_counting()`, a `route()` that sends only counts down that path, and the time each path costs.

> **How to use this notebook**
>
> 1. Run the cells **in order, top to bottom** (Shift+Enter).
>    `# ▶ RUN` cells just run. `# ✏️ EDIT` cells need you to change something first.
> 2. Most steps end with a **✅ Check** cell. If it prints ❌, fix that before moving on. The message says how.
> 3. If the kernel restarts, run the three **Setup** cells again, then carry on from where you were.
> 4. **Turn your VPN off.** The gateway rejects VPN addresses with a 403.

---
## Setup (2 minutes)

Three cells. Run all three before anything else. They are the same in labs 3, 4 and 5, so each notebook works on its own.

Your key is asked for with `getpass`: it is not shown, not saved, and gone when the kernel stops. If you have a `.env` file with `AZURE_OPENAI_KEY` in it, you will not be asked. **Never commit a notebook with a key visible in its output.**

In [1]:
# ▶ RUN  Setup 1 of 3: install the packages into THIS notebook's Python.
# Quick if they are already there. If it installs anything new, restart the
# kernel (Kernel > Restart) and carry on from Setup 2.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ▶ RUN  Setup 2 of 3: connect to the gateway and test all three routes.
import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

try:                                  # optional .env, see .env.example
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# The gateway takes the FULL path as the endpoint, so each client is bound
# to one deployment. The chat route has no /openai segment and the
# embedding route does. That is correct: do not make them match.
CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: a pasted key often carries a trailing newline -> 401.
KEY = (os.getenv("AZURE_OPENAI_KEY") or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

failed = []
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        failed.append(str(e))
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

if not failed:
    print("\n✅ Connected. Carry on.")
elif any("403" in f for f in failed):
    print("\n❌ 403 Forbidden: your VPN is on. Turn it off and run this cell again.")
elif any("401" in f for f in failed):
    print("\n❌ 401: the key is wrong. Run this cell again and paste it carefully.")
else:
    print("\n❌ Not connected. 404 means the URL is wrong (did you edit this cell?). Ask a mentor.")

chat    OK      Linked
vision  OK      connected
embed   OK      1536 dimensions

✅ Connected. Carry on.


In [3]:
# ▶ RUN  Setup 3 of 3: helper functions. Nothing to change here.

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings in batches (one call per string is slow)."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]


# One PersistentClient per folder for the life of the kernel. Opening a
# second one on the same folder corrupts the connection ("attempt to write
# a readonly database"), so rebuilds reset the COLLECTION, not the folder.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    if path not in _stores:
        if reset and Path(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by a "
                f"different chromadb version. Delete that folder and run again.") from None
    client = _stores[path]
    if reset:
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def ensure_index(path="data/chroma", name="workshop"):
    """Open the index lab 2 built. If it is missing or empty, build it from
    sample_corpus/ so this notebook still works if you skipped lab 2."""
    store = get_store(path, name)
    if store.count() == 0:
        print("No lab 2 index found, so building one from sample_corpus/ (about 30 seconds)...")
        texts, metas = [], []
        for f in sorted(Path("sample_corpus").glob("*.txt")):
            for i, piece in enumerate(chunk(f.read_text(encoding="utf-8"))):
                texts.append(piece)
                metas.append({"url": f.name, "position": i, "kind": "text"})
        store = get_store(path, name, reset=True)
        add_to_store(store, texts, metas)
    return store


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks, closest first. Distance is squared L2: lower is closer.
    `where` filters on metadata first. One field: {"kind": "image"}.
    Two or more fields need $and: {"$and": [{"year": "2025"}, {"type": "poster"}]}."""
    r = store.query(query_embeddings=embed([question]), n_results=k, where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Is the expected answer anywhere in the retrieved text? A crude string
    match: read the chunks before you believe a FOUND."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction, sent to the vision deployment.
    There is deliberately no default prompt: writing it is lab 4."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""


# Where each workshop photo was taken. Lab 4 stores this as metadata and
# lab 5 filters on it.
PHOTO_ZONES = {
    "Brainstorming aerial.JPG": "Brainstorming Area",
    "Makerspace close up.JPG":  "Makerspace A",
    "makerspace.JPG":           "Makerspace A",
    "Open Event .JPG":          "Open Event Area",
    "Open event.JPG":           "Open Event Area",
}

print("✅ helpers loaded")

✅ helpers loaded


---
## Step 0 · Check what you are counting (1 minute)

In [4]:
# ▶ RUN
store = ensure_index()
photos = store.get(where={"kind": "image"}, include=["metadatas"])

if not photos["ids"] and Path("data/descriptions.json").exists():
    print("No image chunks in the index, but lab 4 saved its descriptions. Re-indexing them...")
    saved = json.loads(Path("data/descriptions.json").read_text(encoding="utf-8"))
    paths = list(saved)
    add_to_store(store, [saved[p] for p in paths],
                 [{"url": p, "kind": "image", "type": "photo", "zone": PHOTO_ZONES.get(Path(p).name, "unknown")} for p in paths],
                 ids=[f"img_{i}" for i in range(len(paths))])
    photos = store.get(where={"kind": "image"}, include=["metadatas"])

# An older lab 4 did not store zone or type. Add them from the file name.
fix = [(i, m) for i, m in zip(photos["ids"], photos["metadatas"]) if "zone" not in m or "type" not in m]
if fix:
    store.update(ids=[i for i, _ in fix],
                 metadatas=[{**m, "type": "photo", "zone": PHOTO_ZONES.get(Path(m.get("url", "")).name, "unknown")} for _, m in fix])
    photos = store.get(where={"kind": "image"}, include=["metadatas"])

TRUE_COUNT  = len(photos["ids"])
ZONE_COUNTS = {}
for m in photos["metadatas"]:
    ZONE_COUNTS[m.get("zone")] = ZONE_COUNTS.get(m.get("zone"), 0) + 1

if TRUE_COUNT == 0:
    print("❌ No image chunks found. Run lab 4 to the end of Step 4, then run this cell again.")
else:
    print(f"✅ {TRUE_COUNT} photos indexed, by zone: {ZONE_COUNTS}")
    print("   These are the right answers. The rest of the lab checks your bot against them.")

✅ 5 photos indexed, by zone: {'Brainstorming Area': 1, 'Makerspace A': 2, 'Open Event Area': 2}
   These are the right answers. The rest of the lab checks your bot against them.


---
## Step 1 of 5 · Baseline: ask a counting question the normal way (3 minutes)

This is what your bot does today: retrieve the 5 nearest chunks and ask the model. Write down **the answer and the seconds**; you compare against both.

In [5]:
# ▶ RUN
SYSTEM = """Answer only from the context. Reply with the answer only.
If the question asks how many, reply with a number."""

def answer_simple(question, k=5):
    chunks  = query(store, question, k=k)
    context = "\n\n".join(c["text"] for c in chunks)
    return chat([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]).strip()

COUNT_Q = "How many photos of the Innovation Wing have been indexed?"
reply, secs = timed(answer_simple, COUNT_Q)
print(f"Q: {COUNT_Q}\nA: {reply}   ({secs:.1f}s)\n   correct answer: {TRUE_COUNT}")

Q: How many photos of the Innovation Wing have been indexed?
A: 0   (1.6s)
   correct answer: 5


**You should see:** a wrong or lucky number. The 5 nearest chunks are a mix of photos and text pages, and nothing in them says how many photos there are. That is the same failure as the 31 posters.

---
## Step 2 of 5 · Count with a filter (4 minutes)

Three steps, in one function:

1. **Filter, do not search.** `where=` keeps only chunks whose metadata matches, and a big `k` (200) returns all of them, not the nearest 5.
2. **List the evidence.** The model writes out each item before it answers, so a wrong count shows you which item was missed or doubled.
3. **Count the list.** The number goes on its own last line, so code can read it.

This is the function on the "Counting, Step by Step" slide.

In [6]:
# ▶ RUN
def answer_by_counting(question, where, k=200):
    chunks = query(store, question, k=k, where=where)
    print(f"{len(chunks)} items came back  (filter: {where or 'none'}, k={k})")
    listing = "\n".join(f"- {c['text'][:120]}" for c in chunks)
    return chat([
        {"role": "system", "content":
         "Count the items listed. List what you counted, "
         "then give the number on its own final line."},
        {"role": "user", "content": f"Items:\n{listing}\n\nQuestion: {question}"},
    ], max_tokens=700).strip()


def last_number(text):
    """The number on the model's final line, or None."""
    nums = re.findall(r"\d+", (text or "").strip().splitlines()[-1] if text and text.strip() else "")
    return int(nums[-1]) if nums else None

print("functions loaded")

functions loaded


In [7]:
# ▶ RUN  count every photo
reply = answer_by_counting(COUNT_Q, where={"kind": "image"})
print(reply)

5 items came back  (filter: {'kind': 'image'}, k=200)
I counted the following items from your list:

1. Room / area name displayed on a sign: "ENGINEERING" and "INNOVATION"
2. Room or area name shown on a sign: "Open Event Area"
3. Room or area name displayed on any sign: No clearly readable room or area name is visible.
4. Room or area name displayed on any sign: No clear room or area name is displayed on any sign in the image.
5. Room or area name displayed on any sign: [unreadable]

Total count of items: 5

5


In [8]:
# ✅ Check
n = last_number(reply)
if n == TRUE_COUNT:
    print(f"✅ The model counted {n}, and the right answer is {TRUE_COUNT}.")
else:
    print(f"❌ The model said {n}, the right answer is {TRUE_COUNT}. Read its list above: which item did it miss or double?")
print("\nNotice: the filter already returned exactly the right items. When a filter is exact,")
print("len(chunks) IS the answer. The model's list is for checking, and for cases where the filter over-selects.")

✅ The model counted 5, and the right answer is 5.

Notice: the filter already returned exactly the right items. When a filter is exact,
len(chunks) IS the answer. The model's list is for checking, and for cases where the filter over-selects.


Now a narrower count, using the `zone` field lab 4 stored. **One field** goes straight into `where`.

In [9]:
# ▶ RUN  count one zone
reply = answer_by_counting("How many photos of the Open Event Area have been indexed?", where={"zone": "Open Event Area"})
print(reply)
print(f"\n   model: {last_number(reply)}   correct: {ZONE_COUNTS.get('Open Event Area', 0)}")

2 items came back  (filter: {'zone': 'Open Event Area'}, k=200)
I counted the following items:

1. Room or area name shown on a sign
2. "Open Event Area"
3. Room or area name displayed on any sign
4. [unreadable]

Total count of items: 4

4

   model: 4   correct: 2


**Two or more fields** need `$and`. Chroma rejects `{"type": "photo", "zone": "Makerspace A"}` with *"Expected where to have exactly one operator"*.

In [10]:
# ▶ RUN  two fields at once
both = {"$and": [{"type": "photo"}, {"zone": "Makerspace A"}]}
reply = answer_by_counting("How many photos of Makerspace A have been indexed?", where=both)
print(reply)
print(f"\n   model: {last_number(reply)}   correct: {ZONE_COUNTS.get('Makerspace A', 0)}")

2 items came back  (filter: {'$and': [{'type': 'photo'}, {'zone': 'Makerspace A'}]}, k=200)
I counted the following items:

1. Room or area name displayed on any sign
2. No clearly readable room or area name is visible. There is a small sign mount
3. Room or area name displayed on any sign
4. No clear room or area name is displayed on any sign in the image.

Total count: 4

4

   model: 4   correct: 2


---
## Step 3 of 5 · Break it on purpose (2 minutes)

Same question, **no filter** and **k = 3**. The model can only count what it is shown, so the answer can never be more than 3, however good the model is. That is the ceiling from the start of the block, happening in front of you.

In [11]:
# ▶ RUN
reply = answer_by_counting(COUNT_Q, where=None, k=3)
print(reply)
print(f"\n   model: {last_number(reply)}   correct: {TRUE_COUNT}   ceiling: 3")

3 items came back  (filter: none, k=3)
I counted the following items from the list provided:

1. Home
2. About
3. Venues
4. Equipment
5. Funding
6. Programmes
7. Events
8. News
9. Contact
10. Harbour Innovation Centre
11. Faculty of Engineering
12. Gall
13. Inno
14. Brai

Total items counted: 14

14

   model: 14   correct: 5   ceiling: 3


---
## Step 4 of 5 · Design the filter your bot needs for posters (3 minutes)

The real question is *"How many workshop posters are on the Innovation Academy website for 2025?"* A filter can only match fields that were stored when the index was built. First, see which fields your index has.

In [12]:
# ▶ RUN
fields = {}
for m in store.get(include=["metadatas"])["metadatas"]:
    for key, val in m.items():
        fields.setdefault(key, set()).add(str(val))
for key, vals in sorted(fields.items()):
    print(f"  {key:<9} e.g. {', '.join(sorted(vals)[:3])}")

  kind      e.g. image, text
  position  e.g. 0, 1, 2
  type      e.g. photo
  url       e.g. about-centre.txt, about-using-the-centre.txt, contact.txt
  zone      e.g. Brainstorming Area, Makerspace A, Open Event Area


In [13]:
# ✏️ EDIT: fill in the filter that finds 2025 workshop posters, then run.
# Two fields, so it needs $and.
POSTER_FILTER = {"$and": [{"type": "poster"}, {"year": "2025"}]}

In [14]:
# ✅ Check
try:
    matched = len(store.get(where=POSTER_FILTER)["ids"])
    flat = {k: str(v) for part in POSTER_FILTER.get("$and", [POSTER_FILTER]) for k, v in part.items()}
    if flat.get("type") == "poster" and flat.get("year") == "2025":
        print(f"✅ Valid filter. It matches {matched} items here, because the sample corpus has no posters.")
        print("   In your own index it should match all 31, IF every poster chunk stores type and year.")
        print("   Open build/images.py: index_descriptions() stores only url, image and kind. Add type and year there.")
        print('   Store the year as a string, "2025": Chroma compares types exactly, so 2025 and "2025" do not match.')
    else:
        print('❌ Chroma accepted it, but it should filter on type = "poster" and year = "2025".')
except Exception as e:
    print("❌ Chroma rejected the filter:", str(e)[:150])

✅ Valid filter. It matches 0 items here, because the sample corpus has no posters.
   In your own index it should match all 31, IF every poster chunk stores type and year.
   Open build/images.py: index_descriptions() stores only url, image and kind. Add type and year there.
   Store the year as a string, "2025": Chroma compares types exactly, so 2025 and "2025" do not match.


---
## Step 5 of 5 · Route, then time it (3 minutes)

Counting costs time: a big `k`, a long list and a longer answer. Most questions are not counts, so your bot should only take this path when it needs to. **Routing** is one cheap call that classifies the question first.

In [15]:
# ▶ RUN
def route(question):
    lv = ask("Classify this question. Reply with one number only.\n"
             "1 = general knowledge, no lookup needed\n"
             "2 = a single fact from one page\n"
             "3 = a fact that lives in an image\n"
             "4 = needs several facts combined, or a count\n\n"
             f"{question}", max_tokens=4).strip()
    return lv[:1] if lv[:1] in "1234" else "2"

dev = load_dev_set("dev_set.json")
for d in dev + [{"level": 4, "question": COUNT_Q}]:
    print(f"  predicted Lv{route(d['question'])}   actual Lv{d['level']}   {d['question'][:60]}")

  predicted Lv1   actual Lv1   What is 3D printing also known as?
  predicted Lv1   actual Lv1   What does FDM stand for in 3D printing?
  predicted Lv1   actual Lv1   Which faculty would you expect an engineering innovation cen
  predicted Lv2   actual Lv2   When was the deadline for the second round of the Funding Sc
  predicted Lv2   actual Lv2   Which two materials are banned from the laser cutter?
  predicted Lv4   actual Lv3   What were the winning teams in Pitch New Tech Ideas in 2025?
  predicted Lv4   actual Lv4   How many 3D printers does the Centre run in total?
  predicted Lv4   actual Lv4   How many workshop posters are on the Innovation Academy webs
  predicted Lv1   actual Lv5   On the wall at the back of the brainstorming area, there are
  predicted Lv2   actual Lv4   How many photos of the Innovation Wing have been indexed?


In [16]:
# ▶ RUN  what each path costs
lv1 = [d for d in dev if d["level"] == 1][0]["question"]
_, t_simple = timed(answer_simple, lv1)
_, t_count  = timed(answer_by_counting, COUNT_Q, {"kind": "image"})
print(f"\n  normal path    {t_simple:.1f}s   ({lv1[:40]}...)")
print(f"  counting path  {t_count:.1f}s")
print("\nThirty seconds is a hard limit per question. Route first, and only counts pay for counting.")

5 items came back  (filter: {'kind': 'image'}, k=200)

  normal path    1.9s   (What is 3D printing also known as?...)
  counting path  3.1s

Thirty seconds is a hard limit per question. Route first, and only counts pay for counting.


---
## Extension A · When metadata is not enough: retrieve wide, then judge (optional, 4 minutes)

If a poster was never tagged with a year, no filter can find it. The fallback: retrieve wide with **no filter**, let the model mark each item **yes or no**, **drop duplicates**, and **count the yeses in code**. It is still capped by `k`, and it costs one long call, so fixing the metadata is always better.

In [ ]:
# ▶ RUN
def count_by_judging(question, k=20):
    chunks = query(store, question, k=k)
    numbered = "\n".join(f"{i + 1}. {c['text'][:150]}" for i, c in enumerate(chunks))
    reply = ask(f"Question: {question}\n\nFor each numbered item, answer yes if it is one of the "
                f"things the question asks to count, otherwise no. One line per item, like '3: yes'.\n\n{numbered}",
                max_tokens=400)
    yes = {int(m.group(1)) for m in re.finditer(r"(\d+)\s*[:.)-]\s*yes", reply, re.I)}
    kept = [chunks[i - 1] for i in sorted(yes) if 1 <= i <= len(chunks)]
    unique = {c["metadata"].get("url", c["text"][:80]) for c in kept}      # dedupe by source
    print(f"retrieved {len(chunks)}, judged yes {len(kept)}, unique {len(unique)}")
    return len(unique)

n = count_by_judging(COUNT_Q, k=20)
print(f"\n   counted: {n}   correct: {TRUE_COUNT}")

---
## Extension B · A count with two parts (optional, 3 minutes)

*"How many more photos of Makerspace A are there than of the Brainstorming Area?"* is two counts. Count each with its own filter, then do the subtraction **in Python**: models slip on arithmetic. Only split when the count really has two parts; each part is another call.

In [ ]:
# ▶ RUN
a = last_number(answer_by_counting("How many photos of Makerspace A?", where={"zone": "Makerspace A"}))
b = last_number(answer_by_counting("How many photos of the Brainstorming Area?", where={"zone": "Brainstorming Area"}))
print(f"\nMakerspace A: {a}   Brainstorming Area: {b}   difference (computed in code): {None if a is None or b is None else a - b}")
print(f"correct: {ZONE_COUNTS.get('Makerspace A', 0) - ZONE_COUNTS.get('Brainstorming Area', 0)}")

---
## What to take to your own repository

| Where | What |
|---|---|
| `build/images.py`, `index_descriptions()` | store `type` and `year` (and `zone` for your photos) on every chunk. No field, no filter. |
| `bot/answer.py` | `route()` first; for counts, `answer_by_counting()` with the right filter |
| Your experiment log | the seconds each path takes, next to the score |

Deduplicate before you count: the same poster often appears on a listing page and on its own page. Key on the image URL, or 31 becomes 40.